In [46]:
import os
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import contextily as ctx
from statsmodels.tsa.seasonal import seasonal_decompose
from numpy import radians
import folium
from folium.plugins import HeatMap
from windrose import WindroseAxes
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import contextily as ctx
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import gdown

ModuleNotFoundError: No module named 'shapely.lib'

In [44]:
# Load the dataset into a Pandas DataFrame
# Google Drive file ID 
file_id = "1aQmuS8mzgYMDW4vNRIW4uQzpO-GPIMnF"
download_url = f"https://drive.google.com/uc?id={file_id}"

# Download the file
output_file = "hrly_Irish_weather.csv"
gdown.download(download_url, output_file, quiet=False)

# Load CSV into Pandas
data = pd.read_csv(output_file)

# Display the first few rows of the dataset
print(data.head())  

NameError: name 'gdown' is not defined

In [ ]:
# Display basic information about the dataset
print(data.info())

In [ ]:
# Check for missing values
print(data.isnull().sum())

In [ ]:
# Pre-processing and Cleansing
cols_to_convert = ['rain', 'temp', 'wetb', 'dewpt', 'vappr', 'rhum', 'msl', 'wdsp']
data[cols_to_convert] = data[cols_to_convert].apply(pd.to_numeric, errors='coerce')
data['date'] = pd.to_datetime(data['date'], format='%d-%b-%Y %H:%M', errors='coerce')
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['day'] = data['date'].dt.day
data['hour'] = data['date'].dt.hour
data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
geometry = [Point(xy) for xy in zip(data['longitude'], data['latitude'])]
geo_data = gpd.GeoDataFrame(data, geometry=geometry)
scaler = MinMaxScaler()
num_cols = ['rain', 'temp', 'wetb', 'dewpt', 'vappr', 'rhum', 'msl', 'wdsp']
data[num_cols] = scaler.fit_transform(data[num_cols])
data = data.dropna(subset=['rain', 'temp', 'dewpt', 'rhum'])  


In [ ]:
print(data.describe(include='all'))


In [ ]:
print("Shape of the dataset:", data.shape)
print("Columns in the dataset:", data.columns)


In [ ]:
# Counties and Stations
data['county'].value_counts().plot(kind='bar', figsize=(10, 5), title="Data Count per County")
plt.show()


In [ ]:
# Correlation
corr_matrix = data[['rain', 'temp', 'wetb', 'dewpt', 'vappr', 'rhum', 'msl', 'wdsp']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Between Weather Features")
plt.show()


In [ ]:
# Convert longitude and latitude to geometry points
geometry = [Point(xy) for xy in zip(data['longitude'], data['latitude'])]
geo_data = gpd.GeoDataFrame(data, geometry=geometry)

# Set coordinate reference system (CRS) to WGS84 (EPSG:4326)
geo_data.set_crs(epsg=4326, inplace=True)

# Convert to Web Mercator CRS (EPSG:3857) for compatibility with basemaps
geo_data = geo_data.to_crs(epsg=3857)

# Plot on a basemap
fig, ax = plt.subplots(figsize=(10, 10))
geo_data.plot(ax=ax, markersize=10, color='blue', alpha=0.5, label='Stations')

# Add a basemap using OpenStreetMap
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Weather Stations in Ireland")
plt.legend()
plt.show()


In [ ]:
# Geospatial Rainfall Map (Bubble Map)
geo_data['cumulative_rain'] = geo_data.groupby('station')['rain'].transform('sum')
fig, ax = plt.subplots(figsize=(12, 12))
geo_data.drop_duplicates(subset='station').plot(
    ax=ax, markersize=geo_data['cumulative_rain'] / 10, color='cyan', alpha=0.5, legend=True
)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
plt.title("Cumulative Rainfall by Station")
plt.show()


In [ ]:
# Outliers Detection
sns.boxplot(data['rain']).set_title("Rainfall Outliers")
plt.show()


In [ ]:
# Plot temperature over time
plt.figure(figsize=(12, 6))
data['temp'].plot(label="Temperature")
plt.title("Temperature Trends Over Time")
plt.xlabel("Date")
plt.ylabel("Temperature")
plt.legend()
plt.grid()
plt.show()

In [ ]:
data = data[data.index.notnull()]
data['date'] = pd.to_datetime(data['date'], infer_datetime_format=True, errors='coerce')
data.set_index('date', inplace=True)  


In [ ]:
# Rainfall and Temperature Trends Over Time
plt.figure(figsize=(12, 6))
data['temp'].resample('M').mean().plot(label='Average Temperature', color='red')
data['rain'].resample('M').mean().plot(label='Average Rainfall', color='blue')

plt.title("Rainfall and Temperature Trends Over Time")
plt.xlabel("Date")
plt.ylabel("Values")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Convert wddir and wdsp to numeric, coercing errors to NaN
data['wddir'] = pd.to_numeric(data['wddir'], errors='coerce')
data['wdsp'] = pd.to_numeric(data['wdsp'], errors='coerce')

# Drop rows where wddir or wdsp are NaN after conversion
data = data.dropna(subset=['wddir', 'wdsp'])

# Convert wind direction to radians
wddir_radians = np.radians(data['wddir'])

# Plot Wind Speed vs. Wind Direction
plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)
ax.scatter(wddir_radians, data['wdsp'], alpha=0.6, c=data['wdsp'], cmap='viridis')
ax.set_title("Wind Speed vs Wind Direction (Polar Plot)", va='bottom')
plt.show()

In [ ]:
# Wind Rose Plot
ax = WindroseAxes.from_ax()
ax.bar(data['wddir'], data['wdsp'], normed=True, opening=0.8, edgecolor='white')
ax.set_legend()
plt.title("Wind Rose Plot")
plt.show()


In [ ]:
# Histogram of Temperature
plt.figure(figsize=(10, 6))
data['temp'].plot(kind='hist', bins=50, alpha=0.7, color='orange')
plt.title("Histogram of Temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel("Frequency")
plt.grid()
plt.show()


In [ ]:
# Average Rainfall by County
county_rain = data.groupby('county')['rain'].mean()  
county_rain = county_rain.sort_values(ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(x=county_rain.index, y=county_rain.values, palette='Blues_d')
plt.title("Average Rainfall by County")
plt.xlabel("County")
plt.ylabel("Average Rainfall")
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Hourly Trends of Temperature and Rainfall
data['hour'] = data.index.hour  # Extract the hour from the datetime index
hourly_temp = data.groupby('hour')['temp'].mean()
hourly_rain = data.groupby('hour')['rain'].mean()

plt.figure(figsize=(12, 6))
plt.plot(hourly_temp, label="Temperature (°C)", color="red")
plt.plot(hourly_rain, label="Rainfall (mm)", color="blue")
plt.title("Hourly Trends of Temperature and Rainfall")
plt.xlabel("Hour")
plt.ylabel("Values")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Seasonal Trends
temp_decomposed = seasonal_decompose(data['temp'].resample('D').mean(), model='additive', period=365)
temp_decomposed.plot()
plt.show()


In [ ]:
# Scatter Plot: Relation Between Rainfall and Humidity
data = data[~data.index.duplicated()]  
data['rhum'] = pd.to_numeric(data['rhum'], errors='coerce')  
data['rain'] = pd.to_numeric(data['rain'], errors='coerce')  
data = data.dropna(subset=['rhum', 'rain'])  

plt.figure(figsize=(10, 6))
sns.scatterplot(x=data['rhum'], y=data['rain'], alpha=0.5, color='purple')
plt.title("Relationship Between Rainfall and Humidity")
plt.xlabel("Relative Humidity (%)")
plt.ylabel("Rainfall (mm)")
plt.grid()
plt.show()


In [ ]:
# Top Rainfall Days
top_rainfall = data.resample('D')['rain'].sum().nlargest(10)
plt.figure(figsize=(10, 6))
top_rainfall.plot(kind='bar', color='teal', alpha=0.7)
plt.title("Top Rainfall Days")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid()
plt.show()


In [ ]:
# Rainfall Intensity Heatmap
pivot_table = data.pivot_table(index=data.index.hour, columns=data.index.date, values='rain', aggfunc='mean')
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_table, cmap='Blues', cbar_kws={'label': 'Rainfall (mm)'})
plt.title("Hourly Rainfall Intensity Heatmap")
plt.xlabel("Date")
plt.ylabel("Hour of Day")
plt.show()


In [ ]:
# Temperature vs Dew Point (Joint Distribution)
sns.jointplot(x='temp', y='dewpt', data=data, kind='hex', cmap='coolwarm', height=8)
plt.suptitle("Temperature vs Dew Point (Joint Distribution)", y=1.02)
plt.show()


In [ ]:
# Temperature Variance Over the Years
data['year'] = data.index.year
temp_variance = data.groupby('year')['temp'].var()
plt.figure(figsize=(10, 6))
temp_variance.plot(kind='line', color='green', marker='o')
plt.title("Yearly Temperature Variance")
plt.xlabel("Year")
plt.ylabel("Variance")
plt.grid()
plt.show()


In [ ]:
# Boxenplot of Weather Features by County
plt.figure(figsize=(12, 8))
sns.boxenplot(x='county', y='rain', data=data, palette='coolwarm')
plt.title("Rainfall Distribution by County")
plt.xlabel("County")
plt.ylabel("Rainfall (mm)")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Yearly Trends of Rainfall
yearly_rain = data.resample('Y')['rain'].sum()
plt.figure(figsize=(10, 6))
yearly_rain.plot(kind='line', color='blue', marker='o')
plt.title("Yearly Total Rainfall")
plt.xlabel("Year")
plt.ylabel("Total Rainfall (mm)")
plt.grid()
plt.show()


In [ ]:
# Monthly Rainfall Patterns
monthly_rain = data.groupby(data.index.month)['rain'].mean()
plt.figure(figsize=(10, 6))
monthly_rain.plot(kind='bar', color='skyblue')
plt.title("Average Monthly Rainfall")
plt.xlabel("Month")
plt.ylabel("Average Rainfall (mm)")
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid()
plt.show()


In [ ]:
# County-Wise Temperature Trends
county_temp = data.groupby(['county', 'year'])['temp'].mean().unstack(level=0)
county_temp.plot(figsize=(12, 8), colormap='tab10')
plt.title("County-Wise Temperature Trends Over the Years")
plt.xlabel("Year")
plt.ylabel("Average Temperature (°C)")
plt.legend(title='County', loc='upper left')
plt.grid()
plt.show()


In [ ]:
# Rainfall Heatmap (By County and Month)
pivot_table = data.pivot_table(values='rain', index='county', columns=data.index.month, aggfunc='mean')
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_table, cmap='coolwarm', annot=False, cbar_kws={'label': 'Rainfall (mm)'})
plt.title("Average Monthly Rainfall by County")
plt.xlabel("Month")
plt.ylabel("County")
plt.show()


In [ ]:
# Pair Plot for Weather Features
sns.pairplot(data[['temp', 'rain', 'rhum', 'dewpt']], diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle("Pair Plot of Weather Features", y=1.02)
plt.show()


In [ ]:
# Rainfall Intensity by Hour
hourly_rain = data.groupby('hour')['rain'].mean()
plt.figure(figsize=(10, 6))
hourly_rain.plot(kind='bar', color='dodgerblue')
plt.title("Average Hourly Rainfall")
plt.xlabel("Hour of the Day")
plt.ylabel("Rainfall (mm)")
plt.grid()
plt.show()


In [ ]:
# Joint Plot for Wind Speed and Direction
sns.jointplot(x='wddir', y='wdsp', data=data, kind='hex', cmap='viridis')
plt.suptitle("Wind Speed vs Wind Direction (Joint Distribution)", y=1.02)
plt.show()


In [ ]:
# Time Series Decomposition for Rainfall
rain_decomposed = seasonal_decompose(data['rain'].resample('D').mean(), model='additive', period=365)
rain_decomposed.plot()
plt.suptitle("Rainfall Decomposition", y=1.05)
plt.show()


In [ ]:
# Extreme Weather Events
extreme_rainfall = data['rain'].nlargest(20)
plt.figure(figsize=(10, 6))
extreme_rainfall.plot(kind='bar', color='crimson', alpha=0.7)
plt.title("Top 20 Rainfall Events")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid()
plt.show()


In [ ]:
# Select features and target variable
features = ['temp', 'dewpt', 'rhum', 'msl', 'wdsp', 'wddir', 'latitude', 'longitude', 
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
target = 'rain'

# Ensure the required features exist in the dataset
missing_features = [col for col in features if col not in data.columns]
if missing_features:
    raise KeyError(f"The following required features are missing in the dataset: {missing_features}")

X = data[features]
y = data[target]

# Handle missing values
X = X.fillna(X.median())  
y = y.fillna(y.median())  

# Feature scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Check feature statistics
print(X.describe())

# Check correlations of features
sns.heatmap(data[features].corr(), annot=True, cmap='coolwarm')



In [ ]:

# Reshape data for LSTM input (samples, timesteps, features)
X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# Define the LSTM model
model = Sequential([
    LSTM(128, activation='relu', input_shape=(1, X_train.shape[1]), return_sequences=True),
    Dropout(0.3),
    LSTM(64, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])


#Compile the model
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse')

# Train the model
history = model.fit(X_train_lstm, y_train, epochs=50, batch_size=32, validation_data=(X_test_lstm, y_test), callbacks=[
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
])


In [ ]:
# Generate predictions from the test data
y_pred = model.predict(X_test_lstm).flatten()

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)  
mse = mean_squared_error(y_test, y_pred)  
rmse = np.sqrt(mse)  
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100  
r2 = r2_score(y_test, y_pred)  

# Print the evaluation results
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"R-squared (R²): {r2:.2f}")


In [ ]:
# Plot training & validation loss
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()



In [ ]:
# Predefined locations in Ireland
locations_ireland = pd.DataFrame({
    'location': ['Dublin', 'Galway', 'Cork', 'Limerick', 'Belfast'],
    'latitude': [53.3498, 53.2707, 51.8985, 52.6680, 54.5973],
    'longitude': [-6.2603, -9.0568, -8.4756, -8.6305, -5.9301]
})

# Add weather feature averages and cyclical time features for the future date
future_date = datetime.datetime.now() + datetime.timedelta(days=7)
future_month = future_date.month
locations_ireland['date_of_prediction'] = future_date.strftime('%Y-%m-%d')  

# Add the mean values for required weather features from training dataset `data`
locations_ireland['temp'] = data['temp'].mean()
locations_ireland['dewpt'] = data['dewpt'].mean()
locations_ireland['rhum'] = data['rhum'].mean()
locations_ireland['msl'] = data['msl'].mean()
locations_ireland['wdsp'] = data['wdsp'].mean()
locations_ireland['wddir'] = data['wddir'].mean()

# Add cyclical time features
locations_ireland['hour_sin'] = np.sin(2 * np.pi * 12 / 24)  
locations_ireland['hour_cos'] = np.cos(2 * np.pi * 12 / 24)
locations_ireland['month_sin'] = np.sin(2 * np.pi * future_month / 12)
locations_ireland['month_cos'] = np.cos(2 * np.pi * future_month / 12)

# Prepare features for prediction
features = ['temp', 'dewpt', 'rhum', 'msl', 'wdsp', 'wddir', 'latitude', 'longitude',
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
X_ireland = locations_ireland[features]

# Scale features using the trained scaler
X_ireland_scaled = scaler.transform(X_ireland)

# Reshape for LSTM input (samples, timesteps, features)
X_ireland_lstm = X_ireland_scaled.reshape(X_ireland_scaled.shape[0], 1, X_ireland_scaled.shape[1])

# Predict weather for locations
predicted_weather = model.predict(X_ireland_lstm).flatten()

# Reverse scaling for predictions if applied during training
try:
    predicted_weather_actual = scaler.inverse_transform(
        np.concatenate([np.zeros((len(predicted_weather), X_ireland_scaled.shape[1] - 1)),
                        predicted_weather.reshape(-1, 1)], axis=1)
    )[:, -1]
except ValueError:
    # If no scaling was applied to rain, use raw predictions
    predicted_weather_actual = predicted_weather

# Clip and round predictions
predicted_weather_actual = np.clip(predicted_weather_actual, a_min=0, a_max=None).round(6)

# Add predictions to the DataFrame
locations_ireland['predicted_weather_mm'] = predicted_weather_actual

# Final output with consistent weather data
final_output = locations_ireland[['location', 'latitude', 'longitude', 'date_of_prediction',
                                   'predicted_weather_mm', 'temp', 'dewpt', 'rhum', 'msl', 'wdsp']]
print("\nWeather Predictions for Ireland Locations:")
print(final_output)

In [ ]:
# Convert DataFrame with predictions into a GeoDataFrame
geometry = [Point(xy) for xy in zip(locations_ireland['longitude'], locations_ireland['latitude'])]
geo_df = gpd.GeoDataFrame(locations_ireland, geometry=geometry)
geo_df.set_crs(epsg=4326, inplace=True)  # Set CRS to WGS 84 (latitude/longitude)

# Reproject to Web Mercator (EPSG:3857) for plotting with OpenStreetMap
geo_df = geo_df.to_crs(epsg=3857)

# Calculate marker sizes dynamically based on predicted rainfall
geo_df['marker_size'] = geo_df['predicted_weather_mm'] * 50  

# Plot the bubble map
fig, ax = plt.subplots(figsize=(12, 12))
geo_df.plot(
    ax=ax,
    markersize=geo_df['marker_size'],  
    color='cyan',
    alpha=0.6,
    label='Predicted Weather'
)

# Add OpenStreetMap basemap
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

# Annotate each point with weather information
for idx, row in geo_df.iterrows():
    annotation = (
        f"{row['location']} ({row['date_of_prediction']})\n"
        f"Rain: {row['predicted_weather_mm']:.6f} mm\n"
        f"Temp: {row['temp']:.2f}°C\n"
        f"Dewpt: {row['dewpt']:.2f}°C\n"
        f"Rhum: {row['rhum']:.2f}%\n"
        f"Wdsp: {row['wdsp']:.2f} km/h"
    )
    ax.annotate(
        annotation,
        xy=(row.geometry.x, row.geometry.y),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
        color="darkblue"
    )

# Finalize the map
plt.title("Predicted Weather Conditions Across Ireland")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()
